In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import pandas as pd

import firebase_admin
from firebase_admin import credentials, firestore

import streamlit as st  # to access .streamlit/secrets.toml

CSV_PATH = Path("E Commerce Dataset.csv")
COLLECTION_NAME = "current_customers"
BATCH_SIZE = 200  # safe batch size

service_account = st.secrets.get("gcp_service_account", None)

if service_account is None:
    raise RuntimeError("Missing gcp_service_account in .streamlit/secrets.toml")

service_account = dict(service_account)

# Fix newline issue
if "private_key" in service_account:
    service_account["private_key"] = service_account["private_key"].replace("\\n", "\n")

if not firebase_admin._apps:
    cred = credentials.Certificate(service_account)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# helpers
def clean_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass
    return value


def make_doc_id(record: dict, row_idx: int) -> str:
    customer_id = record.get("CustomerID")
    if customer_id is None or str(customer_id).strip() == "":
        return f"seed_{row_idx}"
    return str(customer_id)

#load csv
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")

# ensure no overwriting of existing docs by using batch and checking existence first
batch = db.batch()
batch_count = 0

inserted = 0
skipped = 0
errors = 0

for idx, row in df.iterrows():
    try:
        record = {col: clean_value(row[col]) for col in df.columns}
        record["source"] = "original_dataset"

        doc_id = make_doc_id(record, idx)
        doc_ref = db.collection(COLLECTION_NAME).document(doc_id)

        # Check if doc already exists → skip
        if doc_ref.get().exists:
            skipped += 1
            continue

        batch.set(doc_ref, record)
        batch_count += 1
        inserted += 1

        # Commit in batches
        if batch_count >= BATCH_SIZE:
            batch.commit()
            print(f"Committed batch → inserted: {inserted}, skipped: {skipped}")
            batch = db.batch()
            batch_count = 0

    except Exception as e:
        errors += 1
        print(f"Error on row {idx}: {e}")

# Final commit
if batch_count > 0:
    batch.commit()

print("\n=== DONE ===")
print(f"Inserted: {inserted}")
print(f"Skipped duplicates: {skipped}")
print(f"Errors: {errors}")